# Reproduce charts from exports
Run `python -m sim export` first. Simulated outcomes depend on `config/engine_params.yaml`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
EXPORTS = '../../exports'
metrics = pd.read_parquet(f'{EXPORTS}/metrics.parquet')
runs = pd.read_parquet(f'{EXPORTS}/runs.parquet')
metrics = metrics[metrics.parent_run_id.isna()]  # exclude forks from condition comparisons
metrics.head()

In [ ]:
def bootstrap_ci(values, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return (np.nan, np.nan)
    means = rng.choice(values, (n, len(values))).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))

def by_condition(metric, dimension=''):
    d = metrics[(metrics.metric == metric) & (metrics.dimension == dimension)]
    rows = []
    for (cond, month), g in d.groupby(['condition', 'sim_month']):
        lo, hi = bootstrap_ci(g.value.dropna())
        rows.append({'condition': cond, 'sim_month': month, 'mean': g.value.mean(), 'lo': lo, 'hi': hi})
    return pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (metric, dim) in zip(axes.flat, [('use_cases_live', 'total'), ('ai_revenue_monthly', ''), ('control_count', ''),
                                          ('policy_word_count', ''), ('complaint_rate', ''), ('policy_similarity_cross_bank', '')]):
    t = by_condition(metric, dim)
    for cond, g in t.groupby('condition'):
        ax.plot(g.sim_month, g['mean'], label=cond)
        ax.fill_between(g.sim_month, g.lo, g.hi, alpha=0.2)
    ax.set_title(f'{metric} {dim}'.strip()); ax.tick_params(axis='x', rotation=45)
axes.flat[0].legend(); plt.tight_layout()

In [ ]:
# Kaplan-Meier: months to first MRA-or-worse finding, by condition
findings = pd.read_parquet(f'{EXPORTS}/findings.parquet')
def km(durations, observed):
    order = np.argsort(durations); d, o = np.asarray(durations)[order], np.asarray(observed)[order]
    s, out, at_risk = 1.0, [(0, 1.0)], len(d)
    for t in np.unique(d):
        events = o[d == t].sum()
        if events: s *= 1 - events / at_risk; out.append((t, s))
        at_risk -= (d == t).sum()
    return out
base = runs[runs.parent_run_id.isna()].copy()
first = metrics[metrics.metric == 'first_mra_month_index'].groupby('run_id').value.min()
months_run = metrics.groupby('run_id').sim_month.nunique()
base['observed'] = base.run_id.map(first).notna()
base['duration'] = base.run_id.map(first).fillna(base.run_id.map(months_run))
for cond, g in base.groupby('condition'):
    steps = km(g.duration, g.observed)
    plt.step([x for x, _ in steps], [y for _, y in steps], where='post', label=cond)
plt.legend(); plt.xlabel('months'); plt.ylabel('share without MRA')

In [ ]:
# H4: distance of seat stance from committee median over time (mixed model needs statsmodels)
s = metrics[metrics.metric == 'stance_score'].dropna(subset=['value'])
s['median'] = s.groupby(['run_id', 'sim_month']).value.transform('median')
s['distance'] = (s.value - s['median']).abs()
try:
    import statsmodels.formula.api as smf
    s['t'] = s.groupby('run_id').sim_month.rank(method='dense')
    print(smf.mixedlm('distance ~ t + condition', s, groups=s['replicate']).fit().summary())
except ImportError:
    print('pip install statsmodels for the mixed model'); print(s.groupby('sim_month').distance.mean())